In [ ]:
import pandas as pd
import numpy as np

index_equity: 
    -9 cols, 113 rows
    - no nulls
routes:
    -7 cols, 266 rows
    -no nulls
population: 
    - 7 cols, 9950 rows
    -nulls on 4 cols
    -196 distinct areas
boundaries: 
    - 10 cols, 313 rows
    - nulls on 3 cols
    -313 distinct areas

In [10]:
equity_index = pd.read_csv("./dataset\data\Calgary_Equity_Index_Overlay_Statistics_20260325.csv")
population = pd.read_csv("./dataset\data\Historical_Community_Populations_20260305.csv")
routes = pd.read_csv("./dataset\data\Calgary_Transit_Routes_20260325.csv")
boundaries = pd.read_csv("./dataset\data\community_boundaries.csv")

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not w

In [22]:
boundaries.isnull().sum()

CLASS              0
CLASS_CODE         0
COMM_CODE          0
NAME               0
SECTOR             2
SRG               62
COMM_STRUCTURE     1
CREATED_DT         0
MODIFIED_DT        0
MULTIPOLYGON       0
dtype: int64

In [29]:
pip install rapidfuzz

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 8.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
#auto-matching two datasets by community name in boundaries and population datasets 
#this handles case differences, extra spaces, minor naming variables, and partial matches
from rapidfuzz import process, fuzz
import re

# --- normalization function ---
def normalize(name):
    name = name.upper()
    name = name.strip()
    
    # remove extra spaces around slashes
    name = re.sub(r"\s*/\s*", "/", name)
    
    # remove punctuation (optional)
    name = re.sub(r"[^A-Z0-9/ ]", "", name)
    
    # collapse multiple spaces
    name = re.sub(r"\s+", " ", name)
    
    return name


# normalize datasets
norm1 = {name: normalize(name) for name in population}
norm2 = {name: normalize(name) for name in boundaries}


# reverse lookup for dataset2
norm2_reverse = {v: k for k, v in norm2.items()}


# --- matching ---
matches = []
unmatched = []

THRESHOLD = 85  # adjust (80–90 works well)

for original1, n1 in norm1.items():
    # find best match in dataset2
    result = process.extractOne(
        n1,
        norm2.values(),
        scorer=fuzz.token_sort_ratio
    )
    
    if result:
        match_value, score, _ = result
        
        if score >= THRESHOLD:
            matched_name = norm2_reverse[match_value]
            matches.append((original1, matched_name, score))
        else:
            unmatched.append((original1, score))
    else:
        unmatched.append((original1, 0))


# --- output ---
print("✅ MATCHES:")
for m in matches[:20]:
    print(m)

print("\n❌ UNMATCHED:")
for u in unmatched[:20]:
    print(u)


✅ MATCHES:
('name', 'NAME', 100.0)
('comm_code', 'COMM_CODE', 100.0)

❌ UNMATCHED:
('census_year', 42.10526315789473)
('population', 54.54545454545454)
('occupied dwellings', 35.71428571428571)
('persons per unit', 27.586206896551722)
('notes', 44.44444444444444)


In [33]:
matches_df = pd.DataFrame(matches, columns=["Dataset1", "Dataset2", "Score"])
matches_df

,Dataset1,Dataset2,Score
0,name,NAME,100.0
1,comm_code,COMM_CODE,100.0
